## 3. Model Evaluation Metrics

**Important:** This is an educational example. In production banks:
- Use real payment history (not synthetic)
- Calculate optimal thresholds using economic models
- Consider business costs, risk appetite, regulatory requirements
- Validate thresholds on out-of-time test data

Here we show the methodology for educational purposes.

In [ ]:
import sys
sys.path.insert(0, '/opt/airflow')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    precision_score, recall_score, f1_score, confusion_matrix
)

from src.config import get_db_engine, TrainingConfig, ARTIFACTS_DIR
from src.data.queries import get_feature_dataset
from src.models.scoring.train import get_feature_matrix
from src.models.scoring.artifacts import ScoringArtifact

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)

print('✅ Modules loaded')

## 1. Load models and validation data

In [ ]:
# Load validation data
engine = get_db_engine()
train, val, test = get_feature_dataset(engine, TrainingConfig.TRAIN_END_DATE, TrainingConfig.VAL_END_DATE)

print(f'✅ Validation set: {len(val)} rows, default rate: {val[TrainingConfig.TARGET_COL].mean():.3%}')

# Load models
models = {}
for segment in ['with_history', 'cold_start']:
    path = ARTIFACTS_DIR / segment
    if (path / 'model.pkl').exists():
        models[segment] = ScoringArtifact.load(path)
        print(f'✅ Loaded {segment}: {models[segment].model_type}')

if not models:
    raise RuntimeError('❌ No models found!')

## 2. Make predictions

In [ ]:
# Split by segment and predict
val_with_hist = val[val['avg_days_overdue_90d'].notna() & (val['avg_days_overdue_90d'] != 0)]
val_cold = val[val['avg_days_overdue_90d'].isna() | (val['avg_days_overdue_90d'] == 0)]

results = {}

for segment, artifact in models.items():
    df = val_with_hist if segment == 'with_history' else val_cold
    if len(df) == 0:
        continue
    
    X, _ = get_feature_matrix(df, artifact.feature_names)
    X = X.fillna(pd.Series(artifact.feature_medians))
    
    pred_proba = artifact.predict_proba(X)
    y_true = df[TrainingConfig.TARGET_COL].astype(int).values
    
    results[segment] = {
        'y_true': y_true,
        'y_pred_proba': pred_proba,
        'n_samples': len(df)
    }
    
    print(f'{segment}: {len(df)} samples, mean PD: {pred_proba.mean():.4f}')

## 3. Model Evaluation Metrics

**Important:** This is an educational example. In production banks:
- Use real payment history (not synthetic)
- Calculate optimal thresholds using economic models
- Consider business costs, risk appetite, regulatory requirements
- Validate thresholds on out-of-time test data

Here we show the methodology for educational purposes.

In [ ]:
# Separate plots for ROC-AUC and PR-AUC

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for segment, data in results.items():
    y_true = data['y_true']
    y_pred_proba = data['y_pred_proba']
    
    # ROC curve (LEFT) - for model comparison
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{segment} (AUC={roc_auc:.3f})')
    
    # PR curve (RIGHT) - for business evaluation
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    pr_auc = auc(recall, precision)
    axes[1].plot(recall, precision, label=f'{segment} (PR-AUC={pr_auc:.3f})')

# ROC plot (LEFT)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curves — Model Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR plot (RIGHT)
baseline = results[list(results.keys())[0]]['y_true'].mean()
axes[1].axhline(y=baseline, color='r', linestyle='--', label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves — Business Evaluation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📊 How to use these metrics:')
print('   - ROC-AUC (LEFT): Compare models — which one ranks clients better?')
print('   - PR-AUC (RIGHT): Business evaluation — does model meet requirements?')
print('   - For production: use PR curve + banking-style thresholds (see below)')

## 4. Banking-Style Threshold Selection

**Approach 1: Fixed Recall** — catch X% of defaults

**Approach 2: Fixed Precision** — limit bad approvals

**Approach 3: Cost-based** — minimize business cost

In [ ]:
# Use with_history model for threshold selection
segment = 'with_history'
y_true = results[segment]['y_true']
y_pred_proba = results[segment]['y_pred_proba']

precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)

print('='*70)
print('BANKING-STYLE THRESHOLD SELECTION')
print('='*70)

# Approach 1: Fixed Recall (catch 80% of defaults)
target_recall = 0.80
recall_threshold = None
for i, r in enumerate(recall[:-1]):
    if r >= target_recall:
        recall_threshold = thresholds[i]
        break

if recall_threshold:
    y_pred_recall = (y_pred_proba >= recall_threshold).astype(int)
    precision_at_recall = precision_score(y_true, y_pred_recall)
    print(f'\n📊 Approach 1: Fixed Recall = {target_recall:.0%}')
    print(f'   Threshold: {recall_threshold:.4f}')
    print(f'   Precision: {precision_at_recall:.2%}')
    print(f'   → Catch {target_recall:.0%} of defaults')
    print(f'   → But {1-precision_at_recall:.0%} of approvals are bad')

# Approach 2: Fixed Precision (90% of approvals should be good)
target_precision = 0.90
precision_threshold = None
for i, p in enumerate(precision[:-1]):
    if p >= target_precision:
        precision_threshold = thresholds[i]
        break

if precision_threshold:
    y_pred_precision = (y_pred_proba >= precision_threshold).astype(int)
    recall_at_precision = recall_score(y_true, y_pred_precision)
    print(f'\n📊 Approach 2: Fixed Precision = {target_precision:.0%}')
    print(f'   Threshold: {precision_threshold:.4f}')
    print(f'   Recall: {recall_at_precision:.2%}')
    print(f'   → Only {1-target_precision:.0%} of approvals are bad')
    print(f'   → But miss {1-recall_at_precision:.0%} of defaults')

# Approach 3: Cost-based optimization
cost_fp = 100
cost_fn = 5000

costs = []
for threshold in thresholds[:-1]:
    y_pred = (y_pred_proba >= threshold).astype(int)
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    total_cost = fp * cost_fp + fn * cost_fn
    costs.append(total_cost)

cost_threshold = thresholds[np.argmin(costs)]
y_pred_cost = (y_pred_proba >= cost_threshold).astype(int)
precision_cost = precision_score(y_true, y_pred_cost)
recall_cost = recall_score(y_true, y_pred_cost)

print(f'\n📊 Approach 3: Cost-Based Optimization')
print(f'   Cost FP: ${cost_fp}, Cost FN: ${cost_fn}')
print(f'   Optimal threshold: {cost_threshold:.4f}')
print(f'   Precision: {precision_cost:.2%}, Recall: {recall_cost:.2%}')
print(f'   Min total cost: ${min(costs):,.0f}')

print('\n' + '='*70)
print('RECOMMENDATION:')
print('  Banks typically use Approach 1 or 2 based on risk appetite.')
print('  Approach 3 is optimal but requires accurate cost estimates.')
print('='*70)

## 5. Summary

In [ ]:
print('✅ Threshold Analysis Complete!')
print('\nKey Takeaways:')
print('  1. ROC-AUC is for model comparison, not production decisions')
print('  2. PR-AUC is more honest for imbalanced data')
print('  3. Banks use business-driven thresholds (recall/precision/cost)')
print('  4. Always validate thresholds on test set before production')